https://huggingface.co/blog/afmck/tchaikovsky

OVERALL IMPROVEMENTS: Better usage of GPUs


Used improvements: Use REMI tokenizer instead of standard MIDI tokens

In [1]:
import torch
from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from pathlib import Path
from miditok import REMI, TokenizerConfig
from datasets import Dataset
from tqdm.auto import tqdm # Automatically handles Jupyter vs Console bars
import os
from transformers import default_data_collator
from transformers import TrainingArguments, Trainer

/home/nour_mbb/dl_mbb/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Configuration
# Lakh is huge, so keeping a limit is wise for initial testing
max_files = 3000 

# 2. Updated Path for Lakh Dataset
# Specifically targeting your local directory
dataset_path = Path("/home/nour_mbb/assignment3/datasets/midi/lakh")

print(f"🔍 Searching in: {dataset_path}")

if dataset_path.exists() and dataset_path.is_dir():
    print(f"✅ Found Dataset folder.")
    
    # Lakh often uses .mid or .midi extensions
    # rglob("*.mid*") handles subdirectories recursively
    midi_paths = sorted(list(dataset_path.rglob("*.mid*")))
    
    # Filter to ensure they are files and respect the max_files limit
    midi_paths = [p for p in midi_paths if p.is_file()][:max_files]
else:
    midi_paths = []
    print(f"❌ ERROR: The path {dataset_path} does not exist. Please verify the directory.")

print(f"🎵 Total files found: {len(midi_paths)}")

# 3. Initialize Tokenizer (MidiTok)
config = TokenizerConfig(num_velocities=32, use_chords=True, use_tempos=True)
tokenizer = REMI(config)

# 4. Processing Function
def process_midi_to_dataset(paths):
    all_tokens = []
    if not paths:
        return None
        
    for p in tqdm(paths, desc="Tokenizing Lakh MIDI"):
        try:
            tokens = tokenizer(p) 
            # Note: Lakh is multi-track. tokens[0] takes the first track (usually melody/piano).
            # If you want all tracks merged, you'd need to preprocess the MIDI first.
            if tokens:
                all_tokens.append({"input_ids": tokens[0].ids})
        except Exception as e:
            # Lakh MIDI files are sometimes corrupted or have empty tracks
            continue 
            
    return Dataset.from_list(all_tokens)

# Execute conversion
if midi_paths:
    midi_dataset = process_midi_to_dataset(midi_paths)
    if midi_dataset:
        print(f"Done! Tokenizer Vocabulary Size: {len(tokenizer)}")
        print(f"Dataset size: {len(midi_dataset)} samples")
else:
    print("Process halted: No MIDI files discovered.")

🔍 Searching in: /home/nour_mbb/assignment3/datasets/midi/lakh
✅ Found Dataset folder.
🎵 Total files found: 3000


Tokenizing Lakh MIDI: 100%|█████████████████| 3000/3000 [36:29<00:00,  1.37it/s]


Done! Tokenizer Vocabulary Size: 267
Dataset size: 2961 samples


In [3]:
config = GPT2Config(
    vocab_size=len(tokenizer), 
    n_positions=1024, # Context window
    n_embd=768,       # Hidden dimension for 'Small'
    n_layer=12,       # Layers for 'Small'
    n_head=12,        # Attention heads for 'Small'
    bos_token_id=0,
    eos_token_id=0
)

# Initialize a fresh model with these dimensions
model = GPT2LMHeadModel(config)

# Check model size
print(f"Total Parameters: {model.num_parameters() / 1e6:.2f}M")

Total Parameters: 86.05M


In [4]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3"
print(f"Verfügbare GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"Gerät {i}: {torch.cuda.get_device_name(i)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Verfügbare GPUs: 3
Gerät 0: NVIDIA GeForce GTX 1080
Gerät 1: NVIDIA GeForce GTX 1080
Gerät 2: NVIDIA GeForce GTX 1080


In [5]:
# 1. Split the dataset (80% training, 10% evaluation)
dataset_split = midi_dataset.train_test_split(test_size=0.2)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

In [6]:
training_args = TrainingArguments(
    output_dir="./gpt2-midi-small-lakh",
    overwrite_output_dir=True,
    num_train_epochs=10,
    
    # --- Validation & Printing ---
    eval_strategy="steps",         # Look at validation...
    eval_steps=300,                # ...every 500 steps
    logging_steps=50,              # Print training loss every 50 steps
    
    # --- Best Model Logic ---
    load_best_model_at_end=True,   # Load the best model when finished
    metric_for_best_model="loss",  # Use 'loss' to decide what is "best"
    greater_is_better=False,       # Lower loss is better
    save_strategy="steps",         # Save logic must match eval_strategy
    save_steps=300,
    save_total_limit=2,            # Keep only the best and the most recent
    
    # --- GPU Optimization ---
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    fp16=False,
    report_to="tensorboard"        # Logs the plots
)

In [7]:
# 1. Function to chunk the long MIDI sequences into blocks of 512
def group_midi_tokens(examples, block_size=512):
    # Concatenate all sequences together
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    
    # We drop the small remainder at the end
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
        
    # Split into chunks of block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    
    # GPT-2 needs a 'labels' column (which is just a copy of 'input_ids')
    result["labels"] = result["input_ids"].copy()
    return result

# 2. Apply the chunking to midi_dataset
tokenized_dataset = midi_dataset.map(
    group_midi_tokens,
    batched=True,
    remove_columns=midi_dataset.column_names # Clean up old variable-length rows
)

# 3. Split into Train and Eval
split_ds = tokenized_dataset.train_test_split(test_size=0.1)

# 4. Trainer Call
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=default_data_collator, 
    train_dataset=split_ds["train"],
    eval_dataset=split_ds["test"]
)

trainer.train()

Map: 100%|███████████████████████████| 2961/2961 [03:13<00:00, 15.33 examples/s]
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/home/nour_mbb/dl_mbb/lib/python3.8/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
300,0.429300,0.440998
600,0.389200,0.367288
900,0.342600,0.320019
1200,0.298900,0.296782
1500,0.289800,0.278520
1800,0.294400,0.264029
2100,0.263100,0.251685
2400,0.240700,0.240665
2700,0.246600,0.230293
3000,0.243200,0.221797


/home/nour_mbb/dl_mbb/lib/python3.8/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/nour_mbb/dl_mbb/lib/python3.8/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/nour_mbb/dl_mbb/lib/python3.8/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/nour_mbb/dl_mbb/lib/python3.8/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all in

TrainOutput(global_step=8750, training_loss=0.22562534833635603, metrics={'train_runtime': 34898.2601, 'train_samples_per_second': 24.087, 'train_steps_per_second': 0.251, 'total_flos': 2.1948530688e+17, 'train_loss': 0.22562534833635603, 'epoch': 9.99272052526406})

In [17]:
from pathlib import Path
import random

# 1. Pick a random MIDI file from your list
# We assume 'midi_paths' is the list of paths you created earlier
model.to("cuda:1")
random_midi_path = random.choice(midi_paths)
print(f"🎲 Using random seed file: {random_midi_path.name}")

# 2. Tokenize the seed file
# We take the first 32 tokens to give the model a 'nudge'
seed_tokens = tokenizer(random_midi_path)[0].ids
seed_length = 32 # You can increase this to 64 or 128 for more context
prompt_ids = torch.tensor([seed_tokens[:seed_length]]).to("cuda:1")

# 3. Generate the continuation
# We use top_p (Nucleus Sampling) for better musical variety
output = model.generate(
    prompt_ids,
    max_length=1024,
    temperature=0.8,
    top_p=0.92,
    top_k=0,
    do_sample=True,
    repetition_penalty=1.2,
    pad_token_id=tokenizer['PAD_None']
)

# 4. Extract and Decode
generated_ids = output[0].cpu().numpy().tolist()
print("Generation complete!")

# Convert back to MIDI
generated_midi = tokenizer.tokens_to_midi([generated_ids])

# 5. Save the file
output_filename = f"gen_seed_{random_midi_path.stem}.mid"
generated_midi.dump(output_filename)

print(f"✅ File saved as: {output_filename}")

🎲 Using random seed file: I_Dont_Want_to_Miss_a_Thing.2.mid
Generation complete!
✅ File saved as: gen_seed_I_Dont_Want_to_Miss_a_Thing.2.mid


In [11]:
# Move model to first GPU for inference

model.to("cuda:1")



# Generate 512 tokens (roughly a 1-minute melody)

input_ids = torch.tensor([[0]]).to("cuda:1") # Using BOS token as seed



output = model.generate(

    input_ids,

    max_length=512,

    temperature=0.8,

    top_k=0,

    top_p=0.95,

    do_sample=True,

    repetition_penalty=1.2

)



generated_tokens = output[0].cpu().numpy().tolist()

print("Generation complete!")



generated_midi = tokenizer.tokens_to_midi([generated_tokens])



# 5. Save the MIDI file

output_filename = "generated_piano_piece.mid"

generated_midi.dump(output_filename)



print(f"File saved as: {output_filename}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generation complete!
File saved as: generated_piano_piece.mid
